<a href="https://colab.research.google.com/github/nehansa2003/NLP_project/blob/main/Gradio_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gradio plotly

In [6]:
import gradio as gr
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from google.colab import drive

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
PROJECT_PATH = '/content/drive/MyDrive/Game_Review_NLP'

In [8]:
predictions = pd.read_csv(
    f'{PROJECT_PATH}/all_review_predictions.csv'
)

aspect_analysis = pd.read_csv(
    f'{PROJECT_PATH}/aspect_analysis.csv'
)

key_phrases = pd.read_csv(
    f'{PROJECT_PATH}/key_phrases.csv'
)

game_pros_cons = pd.read_csv(
    f'{PROJECT_PATH}/game_pros_cons.csv'
)

game_key_points = pd.read_csv(
    f'{PROJECT_PATH}/game_key_points.csv'
)

game_comparison = pd.read_csv(
    f'{PROJECT_PATH}/game_comparison.csv'
)

print("All analysis files loaded")

All analysis files loaded


Dashboard analysis function

In [9]:
def dashboard_analysis(game_name):

    sentiment = game_comparison[
        game_comparison['game_name']
        == game_name
    ]

    if sentiment.empty:

        return (
            "Game not found",
            "0",
            "0%",
            "0%",
            "0%",
            None,
            None,
            "No data",
            "No data",
            "No data"
        )

    row = sentiment.iloc[0]

    # -------------------------
    # Sentiment chart
    # -------------------------

    sentiment_fig = go.Figure(
        data=[
            go.Pie(
                labels=[
                    'Positive',
                    'Neutral',
                    'Negative'
                ],
                values=[
                    row['positive_pct'],
                    row['neutral_pct'],
                    row['negative_pct']
                ],
                hole=0.55
            )
        ]
    )

    sentiment_fig.update_layout(
        title='Sentiment Distribution',
        height=400
    )

    # -------------------------
    # Aspect chart
    # -------------------------

    aspects = aspect_analysis[
        aspect_analysis['game_name']
        == game_name
    ].sort_values(
        'mentions',
        ascending=True
    )

    aspect_fig = go.Figure()

    aspect_fig.add_trace(
        go.Bar(
            x=aspects['mentions'],
            y=aspects['aspect'],
            orientation='h'
        )
    )

    aspect_fig.update_layout(
        title='Aspect Frequency',
        height=450
    )

    # -------------------------
    # Pros
    # -------------------------

    pros = game_pros_cons[
        (game_pros_cons['game_name'] == game_name)
        &
        (game_pros_cons['type'] == 'advantage')
    ].head(5)

    pros_text = "\n".join(
        [
            f"• **{x}**"
            for x in pros['aspect']
        ]
    )

    # -------------------------
    # Cons
    # -------------------------

    cons = game_pros_cons[
        (game_pros_cons['game_name'] == game_name)
        &
        (game_pros_cons['type'] == 'disadvantage')
    ].head(5)

    cons_text = "\n".join(
        [
            f"• **{x}**"
            for x in cons['aspect']
        ]
    )

    # -------------------------
    # Key points
    # -------------------------

    points = game_key_points[
        game_key_points['game_name']
        == game_name
    ]

    points_text = "\n".join(
        [
            f"• {x}"
            for x in points['key_point']
        ]
    )

    # -------------------------
    # Key phrases
    # -------------------------

    phrases = key_phrases[
        key_phrases['game_name']
        == game_name
    ]

    positive_phrases = phrases[
        phrases['sentiment'] == 'positive'
    ].head(5)

    negative_phrases = phrases[
        phrases['sentiment'] == 'negative'
    ].head(5)

    phrase_text = (
        "### Positive themes\n"
        +
        "\n".join(
            [
                f"• {x}"
                for x in positive_phrases[
                    'phrase'
                ]
            ]
        )
        +
        "\n\n### Negative themes\n"
        +
        "\n".join(
            [
                f"• {x}"
                for x in negative_phrases[
                    'phrase'
                ]
            ]
        )
    )

    return (

        f"## {game_name}",

        f"{int(row['total_reviews']):,}",

        f"{row['positive_pct']:.1f}%",

        f"{row['neutral_pct']:.1f}%",

        f"{row['negative_pct']:.1f}%",

        sentiment_fig,

        aspect_fig,

        pros_text,

        cons_text,

        points_text,

        phrase_text
    )

app styles

In [10]:
css = """

.gradio-container {
    max-width: 1400px !important;
    margin: auto !important;
    background:
        linear-gradient(
            135deg,
            #0f172a,
            #1e1b4b,
            #111827
        );
}

.title {
    text-align: center;
    font-size: 42px;
    font-weight: 800;
    color: white;
}

.subtitle {
    text-align: center;
    font-size: 18px;
    color: #cbd5e1;
}

.card {
    border-radius: 20px !important;
    padding: 20px !important;
    background:
        rgba(255,255,255,0.08) !important;
    border:
        1px solid rgba(255,255,255,0.15)
        !important;
}

"""

games = sorted(
    game_comparison['game_name']
    .dropna()
    .unique()
)

with gr.Blocks(
    css=css,
    title="Game Review Intelligence"
) as demo:

    gr.Markdown(
        """
        <div class="title">
        🎮 GAME REVIEW INTELLIGENCE
        </div>

        <div class="subtitle">
        NLP-powered video game review analysis
        </div>
        """
    )

    with gr.Row():

        game_dropdown = gr.Dropdown(
            choices=games,
            value=games[0],
            label="🎯 Select a Game",
            interactive=True
        )

    game_title = gr.Markdown()

    # -------------------------
    # SUMMARY
    # -------------------------

    with gr.Row():

        reviews_box = gr.Textbox(
            label="📝 Reviews"
        )

        positive_box = gr.Textbox(
            label="🟢 Positive"
        )

        neutral_box = gr.Textbox(
            label="🟡 Neutral"
        )

        negative_box = gr.Textbox(
            label="🔴 Negative"
        )

    # -------------------------
    # CHARTS
    # -------------------------

    with gr.Row():

        sentiment_plot = gr.Plot(
            label="📊 Sentiment"
        )

        aspect_plot = gr.Plot(
            label="🔍 Themes"
        )

    # -------------------------
    # PROS / CONS
    # -------------------------

    with gr.Row():

        with gr.Column(
            elem_classes="card"
        ):

            gr.Markdown(
                "## 👍 Advantages"
            )

            pros_box = gr.Markdown()

        with gr.Column(
            elem_classes="card"
        ):

            gr.Markdown(
                "## 👎 Disadvantages"
            )

            cons_box = gr.Markdown()

    # -------------------------
    # KEY POINTS
    # -------------------------

    with gr.Row():

        with gr.Column(
            elem_classes="card"
        ):

            gr.Markdown(
                "## 💡 Key Review Points"
            )

            points_box = gr.Markdown()

        with gr.Column(
            elem_classes="card"
        ):

            gr.Markdown(
                "## 🔑 Key Review Themes"
            )

            phrases_box = gr.Markdown()

    # -------------------------
    # GAME CHANGE
    # -------------------------

    outputs = [
        game_title,
        reviews_box,
        positive_box,
        neutral_box,
        negative_box,
        sentiment_plot,
        aspect_plot,
        pros_box,
        cons_box,
        points_box,
        phrases_box
    ]

    game_dropdown.change(
        dashboard_analysis,
        inputs=game_dropdown,
        outputs=outputs
    )

    demo.load(
        dashboard_analysis,
        inputs=game_dropdown,
        outputs=outputs
    )

/tmp/ipykernel_4684/344775114.py:46: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


In [11]:
demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://19719b9c480d8c68a4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://19719b9c480d8c68a4.gradio.live
